# 05 Default State DRLB (Lambda Rule, Optuna 1)

May 04 DRLB run with lambda init rule mapping and Optuna(1) default trial.

May 04 profiles (`may04_*`) set `lambda_min=float('-inf')` and `lambda_max=float('+inf')` so DRLB does not apply finite λ bounds.


In [1]:
import sys
import json
from dataclasses import replace
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd
while repo_root != repo_root.parent and not (repo_root / 'pyproject.toml').exists():
    repo_root = repo_root.parent
if not (repo_root / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate repository root with pyproject.toml')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.shared_runner import run_experiment_inprocess

import importlib
import simulator.model.drlb.state_representations as drlb_state_representations
import simulator.model.drlb.rl_bid_agent_bat as drlb_rl_bid_agent_bat
import simulator.model.drlb_bidder as drlb_bidder_module

importlib.reload(drlb_state_representations)
importlib.reload(drlb_rl_bid_agent_bat)
importlib.reload(drlb_bidder_module)


/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'simulator.model.drlb_bidder' from '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/simulator/model/drlb_bidder.py'>

In [2]:
RUN_NAME = 'may04_default_state_lambda_rule_optuna1'
DRLB_PROFILE = 'may04_default_linear_lambda_legacy'
VERBOSE = False
SHOW_PROGRESS = True


In [3]:
config = build_drlb_config(
    run_name=RUN_NAME,
    profile=DRLB_PROFILE,
    split_set='full_train_val_holdout',
)
config = replace(config, n_trials=10)
config = replace(config, refit_on='train_plus_val')
config = replace(config, max_steps=None)

profile_data = get_drlb_profile(DRLB_PROFILE)
base_drlb_params = dict(profile_data['base_drlb_params'])
reference_model_params = dict(profile_data['reference_model_params'])

with open(repo_root / 'example_notebooks' / 'experiments' / 'drlb' / 'lambda_mapping.json', 'r', encoding='utf-8') as f:
    lambda_mapping = json.load(f)

# Enforce May 04 constraints for lambda rule mode.
reference_model_params['dqn_gamma'] = 1.0
base_drlb_params['init_lambda'] = None
base_drlb_params['init_lambda_mode'] = 'rule'
base_drlb_params['lambda_init_rule'] = lambda_mapping
base_drlb_params['traffic_path'] = str(repo_root / 'data' / 'traffic_share.csv')

from simulator.model.drlb.state_representations import get_state_repr

state_repr = get_state_repr(profile_data['state_type'])
state_repr.begin_episode(1000.0, total_steps=72)
state_vec_len = len(state_repr.curr_state)
if state_vec_len != state_repr.state_size:
    raise RuntimeError(
        f"State representation mismatch for {profile_data['state_type']}: "
        f"len(curr_state)={state_vec_len}, state_size={state_repr.state_size}. "
        "Rerun from the first cell to refresh imports."
    )

result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    show_progress=SHOW_PROGRESS,
    base_drlb_params=base_drlb_params,
    reference_model_params=reference_model_params,
    state_type=profile_data['state_type'],
    objective=profile_data['objective'],
    search_space_fn=lambda trial: {},
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

summary = result['summary']
summary_path = config.outputs_dir / 'run_summary.json'
print(f'Run summary: {summary_path}')
print(json.dumps({
    'run_name': config.run_name,
    'profile': DRLB_PROFILE,
    'state_type': profile_data['state_type'],
    'init_lambda': base_drlb_params['init_lambda'],
    'init_lambda_mode': base_drlb_params['init_lambda_mode'],
    'lambda_init_rule': lambda_mapping,
    'n_trials': config.n_trials,
    'rule_mode_enabled': True,
    'tuning_best_params': summary['tuning']['best_params'],
    'best_val_metrics': summary['tuning']['best_val_metrics'],
    'final_holdout_metrics': summary['final_holdout']['metrics'],
    'diagnostics_png': summary['refit']['combined_diagnostics_plot_path'],
}, indent=2))


autobidder_check campaigns: 100%|██████████| 257/257 [00:15<00:00, 16.52campaign/s, campaign_id=7.46e+7]
[I 2026-05-05 00:08:33,773] A new study created in memory with name: no-name-b1b80f62-cea3-48a5-8237-5545e3fe8b27
Best trial: 0. Best value: 2176.21:  10%|█         | 1/10 [03:26<31:00, 206.71s/it]

[I 2026-05-05 00:12:00,478] Trial 0 finished with value: 2176.213719129918 and parameters: {}. Best is trial 0 with value: 2176.213719129918.


Best trial: 0. Best value: 2176.21:  20%|██        | 2/10 [07:06<28:37, 214.65s/it]

[I 2026-05-05 00:15:40,690] Trial 1 finished with value: 2176.213719129918 and parameters: {}. Best is trial 0 with value: 2176.213719129918.


Best trial: 0. Best value: 2176.21:  30%|███       | 3/10 [10:41<25:03, 214.73s/it]

...

[I 2026-05-05 00:38:42,514] Trial 8 finished with value: 2176.213719129918 and parameters: {}. Best is trial 0 with value: 2176.213719129918.


Best trial: 0. Best value: 2176.21: 100%|██████████| 10/10 [33:22<00:00, 200.26s/it]


[I 2026-05-05 00:41:56,338] Trial 9 finished with value: 2176.213719129918 and parameters: {}. Best is trial 0 with value: 2176.213719129918.


autobidder_check campaigns: 100%|██████████| 257/257 [00:15<00:00, 16.26campaign/s, campaign_id=7.46e+7]


Run summary: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/drlb/may04_default_state_lambda_rule_optuna1/outputs/run_summary.json
{
  "run_name": "may04_default_state_lambda_rule_optuna1",
  "profile": "may04_default_linear_lambda_legacy",
  "state_type": "default",
  "init_lambda": null,
  "init_lambda_mode": "rule",
  "lambda_init_rule": {
    "edges": [
      1.92,
      76.8,
      138.24,
      200.45,
      288.0,
      435.84,
      652.8,
      860.16,
      1254.72,
      2031.36,
      2588.16,
      15465.6
    ],
    "values": [
      0.004362853001546638,
      0.004705136159766192,
      0.004456413834940399,
      0.002661434863468775,
      0.004359150030971928,
      0.0019736742976244055,
      0.0013361573089041045,
      0.000902507841742547,
      0.002398042196987705,
      0.0017812175970161113,
      0.0004418927817928647
    ]
  },
  "n_trials": 10,
  "rule_mode_enabled": true,
  "tuning_best_params": {},
  "best_val_met

In [4]:
rows = [
    {'artifact': 'run_summary_json', 'path': str(config.outputs_dir / 'run_summary.json')},
    {'artifact': 'metrics_json', 'path': str(config.outputs_dir / 'metrics.json')},
    {'artifact': 'drlb_diagnostics_png', 'path': str(config.outputs_dir / 'drlb_diagnostics.png')},
    {'artifact': 'best_refit_model', 'path': str(config.best_models_dir / 'best_refit.pt')},
]
pd.DataFrame(rows)


,artifact,path
0,run_summary_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
1,metrics_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
2,drlb_diagnostics_png,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
3,best_refit_model,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
